# USAD Training Speed Benchmark

Tests three approaches on synthetic data matching the real USAD training dimensions:
- **99,349 windows × 612 features** (same as SWaT after block-median downsample + stride-1 windowing)
- **Batch size 512** → ~194 batches/epoch
- **3 epochs** per trial (enough to benchmark, not full 70)

Approaches:
1. **Current**: CPU TensorDataset + DataLoader, `.item()` per batch (2 syncs/batch)
2. **Device tensor + DataLoader**: move tensor to device before training, `.item()` per batch
3. **Device tensor + manual batching**: no DataLoader, `.item()` once per epoch

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import time

# Detect device
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')

# Synthetic data matching real USAD dimensions
N_WINDOWS  = 99_349
INPUT_SIZE = 612      # W=12 * N_FEATURES=51
Z_DIM      = 40
BATCH_SIZE = 512
N_EPOCHS   = 3        # short run for benchmarking

torch.manual_seed(42)
data_cpu = torch.randn(N_WINDOWS, INPUT_SIZE)  # stays on CPU
data_dev = data_cpu.to(DEVICE)                  # on device

print(f'Windows: {N_WINDOWS:,}  |  Input: {INPUT_SIZE}  |  Batches/epoch: {N_WINDOWS // BATCH_SIZE + 1:,}')

Device: cpu
Windows: 99,349  |  Input: 612  |  Batches/epoch: 195


In [2]:
# USAD model (same architecture as 07-swat-usad.ipynb)
class Encoder(nn.Module):
    def __init__(self, in_size, z_dim):
        super().__init__()
        h1, h2 = in_size // 2, in_size // 4
        self.net = nn.Sequential(
            nn.Linear(in_size, h1), nn.ReLU(),
            nn.Linear(h1, h2),     nn.ReLU(),
            nn.Linear(h2, z_dim),  nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, z_dim, out_size):
        super().__init__()
        h1, h2 = out_size // 4, out_size // 2
        self.net = nn.Sequential(
            nn.Linear(z_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2),    nn.ReLU(),
            nn.Linear(h2, out_size), nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x)

class USAD(nn.Module):
    def __init__(self, in_size, z_dim):
        super().__init__()
        self.E  = Encoder(in_size, z_dim)
        self.D1 = Decoder(z_dim, in_size)
        self.D2 = Decoder(z_dim, in_size)
    def forward(self, w):
        z      = self.E(w)
        w1     = self.D1(z)
        w2     = self.D2(z)
        w2_hat = self.D2(self.E(w1))
        return w1, w2, w2_hat

def l2_mean(a, b):
    return torch.norm(a - b, dim=1).mean()

def make_model():
    torch.manual_seed(42)
    return USAD(INPUT_SIZE, Z_DIM).to(DEVICE)

print('Model defined.')

Model defined.


In [3]:
# ── Approach 1: current — CPU TensorDataset + DataLoader, .item() per batch ──
model = make_model()
opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()))
opt2 = torch.optim.Adam(model.D2.parameters())

ds     = TensorDataset(data_cpu)  # tensor on CPU
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

t0 = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    n = epoch
    ep_t = time.perf_counter()
    for (batch,) in loader:
        batch = batch.to(DEVICE)
        w1, w2, w2_hat = model(batch)
        loss1 = (1/n)*l2_mean(batch, w1) + (1-1/n)*l2_mean(batch, w2_hat)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        w1, w2, w2_hat = model(batch)
        loss2 = (1/n)*l2_mean(batch, w2) - (1-1/n)*l2_mean(batch, w2_hat)
        opt2.zero_grad(); loss2.backward(); opt2.step()
        _ = loss1.item()   # sync per batch (current code)
        _ = loss2.item()   # sync per batch (current code)
    print(f'  Approach 1 | epoch {epoch} | {time.perf_counter()-ep_t:.1f}s')
total1 = time.perf_counter() - t0
print(f'Approach 1 total ({N_EPOCHS} epochs): {total1:.1f}s  |  per epoch: {total1/N_EPOCHS:.1f}s')

  Approach 1 | epoch 1 | 113.1s
  Approach 1 | epoch 2 | 96.1s
  Approach 1 | epoch 3 | 133.7s
Approach 1 total (3 epochs): 343.0s  |  per epoch: 114.3s


In [4]:
# ── Approach 2: device TensorDataset + DataLoader, .item() per batch ──
model = make_model()
opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()))
opt2 = torch.optim.Adam(model.D2.parameters())

ds     = TensorDataset(data_dev)  # tensor already on device
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

t0 = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    n = epoch
    ep_t = time.perf_counter()
    for (batch,) in loader:
        # batch already on device — no .to() needed
        w1, w2, w2_hat = model(batch)
        loss1 = (1/n)*l2_mean(batch, w1) + (1-1/n)*l2_mean(batch, w2_hat)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        w1, w2, w2_hat = model(batch)
        loss2 = (1/n)*l2_mean(batch, w2) - (1-1/n)*l2_mean(batch, w2_hat)
        opt2.zero_grad(); loss2.backward(); opt2.step()
        _ = loss1.item()   # still syncing per batch
        _ = loss2.item()
    print(f'  Approach 2 | epoch {epoch} | {time.perf_counter()-ep_t:.1f}s')
total2 = time.perf_counter() - t0
print(f'Approach 2 total ({N_EPOCHS} epochs): {total2:.1f}s  |  per epoch: {total2/N_EPOCHS:.1f}s')

  Approach 2 | epoch 1 | 25.5s
  Approach 2 | epoch 2 | 67.6s
  Approach 2 | epoch 3 | 120.9s
Approach 2 total (3 epochs): 213.9s  |  per epoch: 71.3s


In [5]:
# ── Approach 3: device tensor + manual batching, .item() once per epoch ──
model = make_model()
opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()))
opt2 = torch.optim.Adam(model.D2.parameters())

t0 = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    n = epoch
    ep_t = time.perf_counter()
    perm = torch.randperm(len(data_dev), device=DEVICE)
    total_l1 = torch.zeros(1, device=DEVICE)
    total_l2 = torch.zeros(1, device=DEVICE)
    n_batches = 0
    for i in range(0, len(data_dev) - BATCH_SIZE + 1, BATCH_SIZE):
        batch = data_dev[perm[i:i+BATCH_SIZE]]
        w1, w2, w2_hat = model(batch)
        loss1 = (1/n)*l2_mean(batch, w1) + (1-1/n)*l2_mean(batch, w2_hat)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        w1, w2, w2_hat = model(batch)
        loss2 = (1/n)*l2_mean(batch, w2) - (1-1/n)*l2_mean(batch, w2_hat)
        opt2.zero_grad(); loss2.backward(); opt2.step()
        total_l1 += loss1.detach()   # no sync
        total_l2 += loss2.detach()   # no sync
        n_batches += 1
    # .item() called ONCE per epoch — one sync
    avg_l1 = (total_l1 / n_batches).item()
    avg_l2 = (total_l2 / n_batches).item()
    print(f'  Approach 3 | epoch {epoch} | {time.perf_counter()-ep_t:.1f}s  l1={avg_l1:.4f} l2={avg_l2:.4f}')
total3 = time.perf_counter() - t0
print(f'Approach 3 total ({N_EPOCHS} epochs): {total3:.1f}s  |  per epoch: {total3/N_EPOCHS:.1f}s')

  Approach 3 | epoch 1 | 23.8s  l1=24.8779 l2=24.8763
  Approach 3 | epoch 2 | 67.2s  l1=24.7289 l2=-0.0001
  Approach 3 | epoch 3 | 131.9s  l1=25.6453 l2=-8.7275
Approach 3 total (3 epochs): 222.9s  |  per epoch: 74.3s


In [ ]:
# ── Summary ──
print(f'{"Approach":<50} {"Per epoch":>10} {"Speedup":>8}')
print('-' * 70)
print(f'1. CPU tensor + DataLoader + .item()/batch         {total1/N_EPOCHS:>9.1f}s {"(baseline)":>8}')
print(f'2. Device tensor + DataLoader + .item()/batch      {total2/N_EPOCHS:>9.1f}s {total1/total2:>7.1f}x')
print(f'3. Device tensor + manual batch + .item()/epoch    {total3/N_EPOCHS:>9.1f}s {total1/total3:>7.1f}x')
print(f'4a. Same + explicit sqrt (no torch.norm)           {total4a/N_EPOCHS:>9.1f}s {total1/total4a:>7.1f}x')
print(f'4b. Same + squared L2 (no sqrt at all in training) {total4b/N_EPOCHS:>9.1f}s {total1/total4b:>7.1f}x')

# Approach 4: replace `torch.norm` with pure arithmetic\n\n`torch.norm(a - b, dim=1)` computes `sqrt(sum((a-b)²))` per sample. On some PyTorch/MPS versions the `sqrt` falls back to CPU, causing a device round-trip every call. We call `l2_mean` **4 times per batch** (twice per forward pass × 2 forward passes), so this matters.\n\nReplacement: `((a - b).pow(2).sum(dim=1) + 1e-8).sqrt().mean()` — all element-wise ops, well-supported on MPS.\n\nFor training the loss, we can also use **squared L2** (`(a - b).pow(2).sum(dim=1).mean()`) which is equivalent for gradient direction (same minimiser) and avoids sqrt entirely. The anomaly score at inference still uses the true L2 norm."

In [ ]:
# ── Approach 4a: device tensor + manual batch + explicit sqrt (no torch.norm) ──
def l2_mean_explicit(a, b):
    # avoids torch.norm which may trigger CPU fallback on MPS
    return ((a - b).pow(2).sum(dim=1) + 1e-8).sqrt().mean()

model = make_model()
opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()))
opt2 = torch.optim.Adam(model.D2.parameters())

t0 = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    n = epoch
    ep_t = time.perf_counter()
    perm = torch.randperm(len(data_dev), device=DEVICE)
    total_l1 = torch.zeros(1, device=DEVICE)
    total_l2 = torch.zeros(1, device=DEVICE)
    n_batches = 0
    for i in range(0, len(data_dev) - BATCH_SIZE + 1, BATCH_SIZE):
        batch = data_dev[perm[i:i+BATCH_SIZE]]
        w1, w2, w2_hat = model(batch)
        loss1 = (1/n)*l2_mean_explicit(batch, w1) + (1-1/n)*l2_mean_explicit(batch, w2_hat)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        w1, w2, w2_hat = model(batch)
        loss2 = (1/n)*l2_mean_explicit(batch, w2) - (1-1/n)*l2_mean_explicit(batch, w2_hat)
        opt2.zero_grad(); loss2.backward(); opt2.step()
        total_l1 += loss1.detach()
        total_l2 += loss2.detach()
        n_batches += 1
    avg_l1 = (total_l1 / n_batches).item()
    avg_l2 = (total_l2 / n_batches).item()
    print(f'  Approach 4a | epoch {epoch} | {time.perf_counter()-ep_t:.1f}s  l1={avg_l1:.4f} l2={avg_l2:.4f}')
total4a = time.perf_counter() - t0
print(f'Approach 4a total ({N_EPOCHS} epochs): {total4a:.1f}s  |  per epoch: {total4a/N_EPOCHS:.1f}s')

In [ ]:
# ── Approach 4b: same but squared L2 (no sqrt at all in training) ──
def sq_l2_mean(a, b):
    # squared L2 per sample — same gradient direction, no sqrt
    return (a - b).pow(2).sum(dim=1).mean()

model = make_model()
opt1 = torch.optim.Adam(list(model.E.parameters()) + list(model.D1.parameters()))
opt2 = torch.optim.Adam(model.D2.parameters())

t0 = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    n = epoch
    ep_t = time.perf_counter()
    perm = torch.randperm(len(data_dev), device=DEVICE)
    total_l1 = torch.zeros(1, device=DEVICE)
    total_l2 = torch.zeros(1, device=DEVICE)
    n_batches = 0
    for i in range(0, len(data_dev) - BATCH_SIZE + 1, BATCH_SIZE):
        batch = data_dev[perm[i:i+BATCH_SIZE]]
        w1, w2, w2_hat = model(batch)
        loss1 = (1/n)*sq_l2_mean(batch, w1) + (1-1/n)*sq_l2_mean(batch, w2_hat)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        w1, w2, w2_hat = model(batch)
        loss2 = (1/n)*sq_l2_mean(batch, w2) - (1-1/n)*sq_l2_mean(batch, w2_hat)
        opt2.zero_grad(); loss2.backward(); opt2.step()
        total_l1 += loss1.detach()
        total_l2 += loss2.detach()
        n_batches += 1
    avg_l1 = (total_l1 / n_batches).item()
    avg_l2 = (total_l2 / n_batches).item()
    print(f'  Approach 4b | epoch {epoch} | {time.perf_counter()-ep_t:.1f}s  l1={avg_l1:.4f} l2={avg_l2:.4f}')
total4b = time.perf_counter() - t0
print(f'Approach 4b total ({N_EPOCHS} epochs): {total4b:.1f}s  |  per epoch: {total4b/N_EPOCHS:.1f}s')